In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr

import sys
sys.path.append("../src")

from yield_spei_perturbation_simulation import *

## Run
One country/case at one time

In [ ]:
dict_planting_date = {
    "id": "11/01",
    "kh": "05/01",
    "mm": "05/01",
    "ph": "06/01",
    "th": "05/01",
    "vn": "05/01" 
}
# growing season set as 10 months from planting to harvest

In [ ]:
country_code = "vn" # select from "id", "kh", "mm", "ph", "th", and "vn"

gid_harvest_area_df = pd.read_csv(
    f"../data/1_harvested_area_processed/harvested_area_{country_code}.csv"
)
gid_harvest_area_df = gid_harvest_area_df[["GID", "sugarcane"]]
gid_harvest_area_df.columns = ["GID", "harvest_area_ha"]

observed_yield_df = pd.read_csv(
    f"../data/3_detrended_yield_processed/"
    f"detrended_yield_sugarcane_{country_code}.csv"
)
observed_yield_df = observed_yield_df[["Year", "DetrendQuadratic"]]
observed_yield_df.columns = ["year", "yield_obs_t_ha"]

gid_harvest_area_selected_df = get_gid_ha_by_coverage(
    gid_harvest_area_df,
    threshold=0.9
)

gdf = gpd.read_file(
    "../data/grid_sea_0p5deg_processed/admin0_grids_0p5deg.gpkg",
    layer=country_code.upper()
)
ds = xr.open_dataset("../data/2_spei_0p5deg_processed/spei3_0p5deg.nc")
da = ds["SPEI3"]

simulation_results_df = simulate_yearly_yield(
    gid_harvest_area_selected_df,
    gdf,
    da,
    observed_yield_df,
    beta=0.25,
    planting_date=dict_planting_date[country_code],
    growing_months=10,
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
    n_trials=30,
    yield_threshold=0.01*observed_yield_df["yield_obs_t_ha"].mean(),
)

In [ ]:
simulation_results_df.head()

In [ ]:
grid_yield_output_path = Path(
    "../outputs/3.2_simulated_yield/d_sugarcane/"
    "sugarcane_grid_yield_vn_beta0p25_260720.csv"
)

simulation_results_df = simulation_results_df.round(
    {
        "harvest_area_ha": 2,
        "spei": 4,
        "mean_perturbation": 4,
        "yield_sim_t_ha": 4,
        "scale_factor": 4,
        "yield_scaled_t_ha": 4
    }
)

simulation_results_df.to_csv(grid_yield_output_path, index=False)